# Paper tables

The analysis uses the photo offer recorded in the archived source links for each property.
The selected-offer mapping and the exterior and floor-plan technical metrics are read from local Parquet caches.

In [1]:
from pathlib import Path
import sys

working_dir = Path.cwd()
root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "inputs" / "prompts.json").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the final_project directory or one of its subdirectories.")
sys.path.insert(0, str(root / "code"))

In [2]:
from IPython.display import display
import pandas as pd

from utils.paper_data import DATASETS_DIR, MARKETS, load_analysis_data

tables_dir = root / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
listings, photo_types, quality_models_by_type = load_analysis_data(root)
interiors = photo_types["interior"]
quality_models = quality_models_by_type["interior"]


def save_table(table, number, **options):
    display(table)
    path = tables_dir / f"table_{number}.tex"
    path.write_text(table.to_latex(float_format="{:.3f}".format, escape=True, na_rep="", **options), encoding="utf-8")


def coefficient_cell(model, term):
    if term not in model.params:
        return ""
    p = model.pvalues[term]
    stars = "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""
    return f"{model.params[term]:.3f}{stars}"

## Methods and descriptives

Tables 1–4. The listing summaries use the current photo eligibility.
Image descriptives and the image-quality regression use the same finished-interior sample.

### Table 1. CLIP polarity calculation

In [3]:
steps = pd.DataFrame({"Calculation": [
    "Use the saved similarities for the positive, neutral, negative, and unfinished prompts.",
    "Multiply the similarities by the learned CLIP scale to obtain logits.",
    "Apply softmax across the four logits to obtain class probabilities.",
    "Flag interiors as unfinished when their unfinished probability is at least 0.90.",
    "For finished photos, condition on the three finished classes and subtract the negative probability from the positive probability.",
]}, index=pd.Index(["Step 1", "Step 2", "Step 3", "Step 4", "Step 5"], name="Step"))
save_table(steps, "01", column_format=r"lp{0.73\linewidth}")

,Calculation
Step,
Step 1,"Use the saved similarities for the positive, n..."
Step 2,Multiply the similarities by the learned CLIP ...
Step 3,Apply softmax across the four logits to obtain...
Step 4,Flag interiors as unfinished when their unfini...
Step 5,"For finished photos, condition on the three fi..."


### Table 2. Sample means

In [4]:
categorical = {
    "roomsCount_grouped": "Number of rooms",
    "bathrooms_cat": "Number of bathrooms",
    "ceilingHeightCapped": "Ceiling height (m)",
    "apartment": "Apartment",
    "toDemolish": "Demolition program",
    "floorNumberCapped": "Floor number",
    "material": "Construction material",
    "repairType": "State of repair",
}
numerical = {
    "totalArea": "Area (square metres)",
    "age": "Building age (years)",
    "distance_center": "Distance to center",
    "distance_metro": "Distance to metro",
    "distance_nice": "Distance to green area or water",
    "distance_ugly": "Distance to industrial zone",
}
means = {}
for market, title in MARKETS.items():
    data = listings.loc[listings["deal_type"].eq(market) & listings["link_num"].gt(0)]
    rows = {(label, "Mean"): data[column].mean() for column, label in numerical.items()}
    for column, label in categorical.items():
        shares = data[column].value_counts(normalize=True, sort=False)
        if column in ("apartment", "toDemolish"):
            shares = shares.loc[["yes"]]
        rows.update({(label, level): share for level, share in shares.items()})
    rows["Listings with usable photos", "N"] = f"{len(data):,}"
    means[title] = pd.Series(rows)

table = pd.DataFrame(means)
table.index.names = ["Variable", "Category"]
# Geographic distances are in units of 100 metres.
save_table(table, "02")

Long-term Rentals Secondary Sales
Variable                        Category                                  
Area (square metres)            Mean             54.852406       61.845447
Building age (years)            Mean             32.637999       29.612184
Distance to center              Mean            111.031193      109.099358
Distance to metro               Mean              7.032753        7.212756
Distance to green area or water Mean              7.081138        6.730674
Distance to industrial zone     Mean              5.598996        5.414793
Number of rooms                 1                 0.381044        0.286459
                                2                 0.403759        0.380358
                                3                 0.181032         0.26076
                                4+                0.034165        0.072423
Number of bathrooms             1                  0.87062        0.787762
                                2+                 0.12938        0.212238
Ceiling height (m)              2.4-2.7           0.178667        0.239464
                                2.7-3.0            0.15477         0.19283
                                3.0+              0.106603         0.26112
                                NR                 0.55996        0.306586
Apartment                       yes                0.03877        0.064876
Demolition program              yes               0.051714        0.022823
Floor number                    1st                0.05047        0.055081
                                2-9               0.606012        0.578399
                                10-19             0.265044        0.264983
                                20-29             0.057315        0.067212
                                30+               0.021159        0.034325
Construction material           brick             0.164665         0.16165
                                block             0.080839        0.071974
                                panel             0.330948        0.305598
                                concrete          0.225714        0.353581
                                NR                0.197834        0.107197
State of repair                 Designer          0.225278        0.228682
                                Euro              0.381106        0.210621
                                Cosmetic          0.301263        0.300297
                                None              0.011264        0.188337
                                NR                0.081088        0.072064
Listings with usable photos     N                   16,069          11,129

### Table 3. Image quality descriptives

In [5]:
image_columns = {
    "ratio": "Aspect ratio",
    "resolution": "Resolution (million pixels)",
    "sharpness": "Sharpness (thousands)",
    "brightness": "Brightness",
    "brisque": "BRISQUE",
    "clip_polarity": "CLIP polarity",
}
finished = interiors.dropna(subset=["clip_polarity_adj"])
quality_stats = {}
for market, title in MARKETS.items():
    photos = finished.loc[finished["deal_type"].eq(market)]
    stats = photos[list(image_columns)].describe().loc[["mean", "std", "min", "50%", "max"]].T
    quality_stats[title] = stats.rename(index=image_columns, columns={"50%": "median"})
    print(f"{title}  N = {len(photos):,}")
save_table(pd.concat(quality_stats, axis=1), "03")

Long-term Rentals  N = 229,033
Secondary Sales  N = 143,141


Long-term Rentals            ... Secondary Sales           
                                         mean       std  ...          median        max
Aspect ratio                         1.018106  0.382397  ...        0.892708   4.785714
Resolution (million pixels)          0.820030  0.248763  ...        0.691200   1.637120
Sharpness (thousands)                0.547860  0.719775  ...        0.367064  10.000000
Brightness                           0.514412  0.088669  ...        0.512434   0.986734
BRISQUE                              0.266395  0.124656  ...        0.257380   1.000000
CLIP polarity                        0.230453  0.316443  ...        0.184491   0.976959

[6 rows x 10 columns]

### Table 4. Regression of CLIP polarity on image quality

In [6]:
results = {}
for market, model in quality_models.items():
    rows = {}
    for term, label in image_columns.items():
        if term in model.params:
            rows[label] = coefficient_cell(model, term)
            rows[f"{label} (clustered SE)"] = f"({model.bse[term]:.3f})"
    rows["Observations"] = f"{model.nobs:,.0f}"
    rows["R²"] = f"{model.rsquared:.3f}"
    rows["F-statistic"] = f"{model.fvalue:.1f}"
    results[MARKETS[market]] = rows
save_table(pd.DataFrame(results), "04")

,Long-term Rentals,Secondary Sales
Aspect ratio,0.044***,0.096***
Aspect ratio (clustered SE),(0.006),(0.010)
Resolution (million pixels),0.060***,0.016
Resolution (million pixels) (clustered SE),(0.009),(0.014)
Sharpness (thousands),0.018***,0.033***
Sharpness (thousands) (clustered SE),(0.002),(0.004)
Brightness,0.607***,0.775***
Brightness (clustered SE),(0.017),(0.025)
BRISQUE,0.201***,0.360***
BRISQUE (clustered SE),(0.010),(0.015)


## Price models and time on market

Tables 5–7 and the model-based numbers use listings with adjusted interior Q
and the required listing data. The price models share a train/test split
within each market. Model 1 uses listing controls. Model 2 adds adjusted
interior Q.

### Fit OLS and histogram gradient boosting

In [7]:
from utils.paper_data import load_llama_scores
from utils.paper_models import (
    PHOTO_FEATURES, FEATURE_LABELS,
    boost_features, fit_price_models, model_performance, ols_features,
)
from utils.paper_survival import duration_sample

fits = {market: fit_price_models(listings, market) for market in MARKETS}
durations = {market: duration_sample(listings, market) for market in MARKETS}

### Tables 5 and 6. Hedonic regression and gradient boosting

In [8]:
ols_columns, hgb_columns = {}, {}
test_performance = {}
for market, (train, test, models) in fits.items():
    for number, model in models.items():
        key = (MARKETS[market], f"({number})")
        ols = model["ols"]
        ols_rows = {}
        for term in PHOTO_FEATURES:
            label = FEATURE_LABELS[term]
            ols_rows[label] = coefficient_cell(ols, term)
            ols_rows[f"{label} (HC3 SE)"] = f"({ols.bse[term]:.3f})" if term in ols.params else ""
        hgb_rows = {}
        for split, data in [("Train", train), ("Test", test)]:
            actual = data["log_price"]
            ols_prediction = ols.predict(ols_features(model["design_info"], data))
            hgb_prediction = model["hgb"].predict(boost_features(data, model["features"]).to_numpy())
            ols_stats = model_performance(actual, ols_prediction, int(ols.df_model))
            hgb_stats = model_performance(actual, hgb_prediction, len(model["features"]))
            if split == "Test":
                test_performance[market, number] = {"ols": ols_stats, "hgb": hgb_stats}
            for label, result in ols_stats.items():
                ols_rows[f"{split} {label}"] = f"{result:,.0f}" if label == "N" else f"{result:.3f}"
            for label, result in hgb_stats.items():
                hgb_rows[f"{split} {label}"] = f"{result:,.0f}" if label == "N" else f"{result:.3f}"
        for rows in (ols_rows, hgb_rows):
            rows["Original listing features"] = "Yes"
            rows["Interior Q-score"] = "Yes" if number == 2 else ""
        ols_columns[key] = ols_rows
        hgb_columns[key] = hgb_rows

save_table(pd.DataFrame(ols_columns), "05")
save_table(pd.DataFrame(hgb_columns), "06")

Long-term Rentals  ... Secondary Sales
                                                 (1)  ...             (2)
Adjusted interior Q-score                             ...        0.200***
Adjusted interior Q-score (HC3 SE)                    ...         (0.011)
Train N                                       12,629  ...           8,294
Train Adjusted R²                              0.872  ...           0.912
Train RMSE                                     0.179  ...           0.180
Test N                                         3,158  ...           2,074
Test Adjusted R²                               0.865  ...           0.896
Test RMSE                                      0.186  ...           0.197
Original listing features                        Yes  ...             Yes
Interior Q-score                                      ...             Yes

[10 rows x 4 columns]

Long-term Rentals         Secondary Sales       
                                        (1)     (2)             (1)    (2)
Train N                              12,629  12,629           8,294  8,294
Train Adjusted R²                     0.923   0.933           0.965  0.961
Train RMSE                            0.140   0.130           0.115  0.121
Test N                                3,158   3,158           2,074  2,074
Test Adjusted R²                      0.886   0.900           0.915  0.919
Test RMSE                             0.174   0.163           0.184  0.180
Original listing features               Yes     Yes             Yes    Yes
Interior Q-score                                Yes                    Yes

### Table 7. Time on the market

In [9]:
summaries = {}
for market, data in durations.items():
    summary = data.groupby("status")["dur"].agg(["count", "mean", "std", "min", "median", "max"])
    summary = summary.reindex([True, False]).rename(index={True: "Closed", False: "Censored"})
    summary["count"] = summary["count"].map(lambda n: f"{n:,.0f}")
    summaries[MARKETS[market]] = summary.round(1)
save_table(pd.concat(summaries, names=["Market", "Status"]), "07")

count   mean   std   min  median    max
Market            Status                                            
Long-term Rentals Closed    15,125   27.6  30.1   1.2    16.5  156.6
                  Censored     602  126.5  17.7  60.5   126.5  156.6
Secondary Sales   Closed     8,496   51.6  37.2   1.3    41.3  157.8
                  Censored   1,840  128.6  16.9  60.0   128.5  157.4

## Appendix

### Table A.1. LLaMA4 image-rating prompt

In [10]:
prompts = pd.Series({
    "System prompt": "You are a helpful assistant that can look at pictures of real estate and provide accurate descriptions and analysis. Answer the user's questions regarding the attached images as clearly and accurately as possible.",
    "User query": "Rate how luxurious and upscale this apartment looks based on the pictures. Give a score from 0 to 10 to each photo. Answer with a list of scores (e.g.: '[4,3,5,7,2,0,9]'). Omit any explanation or additional information.",
    "Image inputs": "Up to ten photo URLs, randomly selected when a listing has more than ten photos.",
    "Decoding": "Temperature 0",
}, name="Prompt or setting")
save_table(prompts.to_frame(), "A1", column_format=r"lp{0.73\linewidth}")

,Prompt or setting
System prompt,You are a helpful assistant that can look at p...
User query,Rate how luxurious and upscale this apartment ...
Image inputs,"Up to ten photo URLs, randomly selected when a..."
Decoding,Temperature 0


## Computed numbers cited in the text

In [11]:
llama_path = root / DATASETS_DIR / "llama4_scores.parquet"
llama = load_llama_scores(llama_path, listings) if llama_path.is_file() else None
numbers = []
for market in MARKETS:
    data = listings.loc[listings["deal_type"].eq(market)]
    photos = interiors.loc[interiors["deal_type"].eq(market)]
    usable = data["link_num"].gt(0)
    train, test, models = fits[market]
    duration = durations[market]
    metrics = {
        "listings": len(data),
        "first_listing_date": data["date0"].min().date().isoformat(),
        "last_listing_date": data["date0"].max().date().isoformat(),
        "last_observed_date": data["date1"].max().date().isoformat(),
        "mean_new_listings_per_day": data.groupby(data["date0"].dt.date).size().mean(),
        "owner_share": data["author_cat"].eq("Owner").mean(),
        "fraction_without_usable_photos": (~usable).mean(),
        "listings_with_usable_photos": usable.sum(),
        "usable_photos": data["link_num"].sum(),
        "interior_photos": len(photos),
        "landscape_share_of_interiors_with_dimensions": photos["ratio"].dropna().gt(1).mean(),
        "jpeg_interior_photos": photos["format"].eq("JPEG").sum(),
        "grayscale_interior_photos": photos["mode"].eq("L").sum(),
        "unfinished_interior_photos": photos["unfinished"].sum(),
        "listings_with_all_scored_interiors_unfinished": data["share_unfinished"].eq(1).sum(),
        "finished_interiors_with_quality_metrics": int(quality_models[market].nobs),
        "fraction_polarity_variance_unexplained": 1 - quality_models[market].rsquared,
        "listings_with_adjusted_interior_score": data["clip_polarity_adj"].count(),
        "listings_with_exterior_score": data["exterior_polarity"].count(),
        "listings_with_floor_plan_score": data["plan_polarity"].count(),
        "price_train_listings": len(train),
        "price_test_listings": len(test),
        "ols_interior_price_percent_change_per_point_one": 100 * 0.1 * models[2]["ols"].params["clip_polarity_adj"],
        "duration_listings": len(duration),
        "right_censored_share": duration["status"].eq(False).mean(),
    }
    if llama is not None:
        ratings = llama.loc[llama["deal_type"].eq(market) & llama["link_num"].gt(0)]
        comparable = ratings[["clip_polarity", "average_luxury_score"]].dropna()
        metrics["llama_scored_listings"] = ratings["average_luxury_score"].count()
        metrics["llama_scored_share"] = ratings["average_luxury_score"].notna().mean()
        metrics["clip_llama_comparable_listings"] = len(comparable)
        metrics["clip_llama_pearson_correlation"] = comparable["clip_polarity"].corr(comparable["average_luxury_score"])
    for kind in ("ols", "hgb"):
        baseline = test_performance[market, 1][kind]["Adjusted R²"]
        metrics[f"{kind}_test_adj_r2_gain_interior"] = test_performance[market, 2][kind]["Adjusted R²"] - baseline
    numbers.extend((label, market, value) for label, value in metrics.items())

numbers = pd.DataFrame(numbers, columns=["metric", "deal_type", "value"])
numbers.to_csv(tables_dir / "paper_numbers.csv", index=False)
display(numbers.pivot(index="metric", columns="deal_type", values="value"))

deal_type,long_rent,sale_secondary
metric,,
clip_llama_comparable_listings,14090,9657
clip_llama_pearson_correlation,0.780098,0.726596
duration_listings,15727,10336
finished_interiors_with_quality_metrics,229033,143141
first_listing_date,2025-07-01,2025-07-01
fraction_polarity_variance_unexplained,0.947573,0.917928
fraction_without_usable_photos,0.06841,0.054621
grayscale_interior_photos,0,0
hgb_test_adj_r2_gain_interior,0.013987,0.003343


## Photo-score variants

A is the current interior specification on the source-selected photo offers. B and C use exterior and floor-plan Q respectively. D includes all three Q-scores, E uses mean unfinished probability, and F includes all four scores.

The suffixed tables use each variant’s eligible sample. The two shared-sample tables hold the listing cohort fixed within each market and outcome.

In [12]:
from utils.paper_variant_tables import export_variant_tables

shared_price, shared_duration = export_variant_tables(
    tables_dir, listings, photo_types, quality_models_by_type, llama, fits, durations,
)
display(shared_price.round(3))
display(shared_duration.round(3))

Saved Tables 01 and 03–07 for variant B
Saved Tables 01 and 03–07 for variant C
Saved Tables 01 and 03–07 for variant D
Saved Tables 01 and 03–07 for variant E
Saved Tables 01 and 03–07 for variant F


,Market,Variant,Train N,Test N,OLS adjusted R²,OLS RMSE,Boosting adjusted R²,Boosting RMSE
0,Long-term Rentals,baseline,3317,830,0.880,0.168,0.899,0.168
2,Long-term Rentals,A,3317,830,0.901,0.153,0.911,0.158
4,Long-term Rentals,B,3317,830,0.887,0.163,0.903,0.165
6,Long-term Rentals,C,3317,830,0.880,0.168,0.897,0.170
8,Long-term Rentals,D,3317,830,0.902,0.151,0.913,0.156
10,Long-term Rentals,E,3317,830,0.880,0.168,0.901,0.167
12,Long-term Rentals,F,3317,830,0.902,0.151,0.912,0.157
1,Secondary Sales,baseline,4628,1157,0.903,0.182,0.925,0.170
3,Secondary Sales,A,4628,1157,0.909,0.177,0.933,0.162
5,Secondary Sales,B,4628,1157,0.905,0.180,0.925,0.170


,Market,Variant,Score,N,Closed,Censored,Concordance,HR (+1 point),Lower 95%,Upper 95%
0,Long-term Rentals,baseline,,4139,3995,144,0.619,NaN,NaN,NaN
2,Long-term Rentals,A,Adjusted interior Q-score,4139,3995,144,0.623,1.542,1.223,1.945
4,Long-term Rentals,B,Adjusted exterior Q-score,4139,3995,144,0.619,0.973,0.861,1.101
6,Long-term Rentals,C,Adjusted floor-plan Q-score,4139,3995,144,0.619,0.920,0.738,1.146
8,Long-term Rentals,D,Adjusted interior Q-score,4139,3995,144,0.623,1.578,1.247,1.997
9,Long-term Rentals,D,Adjusted exterior Q-score,4139,3995,144,0.623,0.927,0.818,1.051
10,Long-term Rentals,D,Adjusted floor-plan Q-score,4139,3995,144,0.623,0.953,0.764,1.190
14,Long-term Rentals,E,Mean unfinished probability,4139,3995,144,0.619,1.021,0.706,1.476
16,Long-term Rentals,F,Adjusted interior Q-score,4139,3995,144,0.623,1.590,1.253,2.019
17,Long-term Rentals,F,Adjusted exterior Q-score,4139,3995,144,0.623,0.927,0.818,1.051
